# OpenBench Phase 5: TextCNN and Training Engine

This notebook reviews the implemented TextCNN, training workflow, saved experiment, and raw-text inference interface. It loads the completed run; it does not start training.

The experiment used the labeled IMDb training split, divided into 20,000 training and 5,000 validation reviews with seed 42. The vocabulary was fitted on training reviews only. The official IMDb test set and the unlabeled reviews were not loaded or used.

## 1. Architecture and training choices

The model embeds token IDs into 100-dimensional vectors, applies parallel one-dimensional convolution branches with widths 3, 4, and 5, max-pools each branch, then uses dropout and a two-logit classifier. Convolution windows that overlap right-padding are masked before pooling. A review shorter than a filter contributes a zero vector for that branch; empty reviews are supported.

Training uses Adam (learning rate 0.001) and cross-entropy. Validation reports loss, accuracy, positive-class precision, recall, and F1. Checkpoints are selected by the best observed validation F1. Early stopping uses patience 3 and minimum improvement 0.001. The saved run used CUDA for training and was reloaded on CPU for inference verification.

In [1]:
from pathlib import Path
import json
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

candidate_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next(
    path for path in candidate_roots
    if (path / "pyproject.toml").is_file() and (path / "src/deeptext").is_dir()
)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

EXPERIMENT_ID = "imdb-textcnn-20261030T122328.015737Z"
RUN_DIRECTORY = PROJECT_ROOT / "outputs" / "textcnn" / EXPERIMENT_ID
DATASET_DIRECTORY = PROJECT_ROOT / "datasets" / "aclImdb"
assert RUN_DIRECTORY.is_dir(), f"Experiment artifacts not found: {RUN_DIRECTORY}"
print(f"Project: {PROJECT_ROOT}")
print(f"Experiment: {RUN_DIRECTORY}")
print(f"Python: {sys.version.split()[0]} | PyTorch: {torch.__version__}")

Project: C:\Users\thush\Documents\OpenBench
Experiment: C:\Users\thush\Documents\OpenBench\outputs\textcnn\imdb-textcnn-20261030T122328.015737Z
Python: 3.15.2 | PyTorch: 2.15.0+cu132


## 2. Load saved run configuration and history

The JSON files are the source of the displayed metrics. This cell also checks the recorded split policy and the checkpoint reload verification.

In [2]:
metadata = json.loads((RUN_DIRECTORY / "metadata.json").read_text(encoding="utf-8"))
history = json.loads((RUN_DIRECTORY / "history.json").read_text(encoding="utf-8"))["epochs"]
history_frame = pd.DataFrame(history)

assert metadata["official_test_evaluated"] is False
assert metadata["unlabeled_reviews_used"] is False
assert metadata["vocabulary_fit_partition"] == "training"
assert metadata["checkpoint_reload_verified"] is True
assert metadata["reload_probabilities_match"] is True
assert metadata["reload_predictions_match"] is True

display(pd.DataFrame([
    {"setting": key, "value": value}
    for key, value in metadata["model_config"].items()
]))
display(history_frame[[
    "epoch", "training_loss", "validation_loss",
    "training_f1", "validation_f1",
]])
print(f"Best epoch: {metadata['best_epoch']}")
print(f"Best validation metrics: {metadata['best_validation_metrics']}")
print(f"Early stopping: {metadata['stopping_reason']}")

,setting,value
0,dropout,0.5
1,embedding_dim,100
2,filter_sizes,"[3, 4, 5]"
3,filters_per_size,100
4,max_sequence_length,200
5,padding_idx,0
6,vocab_size,30000


,epoch,training_loss,validation_loss,training_f1,validation_f1
0,1,0.470836,0.357107,0.769358,0.841235
1,2,0.277406,0.345942,0.886922,0.855942
2,3,0.142978,0.373494,0.948655,0.853280
3,4,0.052735,0.479746,0.983761,0.849743
4,5,0.021751,0.580945,0.993947,0.848340


Best epoch: 2
Best validation metrics: {'accuracy': 0.8492, 'f1': 0.8559419182269774, 'loss': 0.34594207081794737, 'precision': 0.8193123628383321, 'recall': 0.896}
Early stopping: Validation F1 did not improve by more than 0.001 for 3 consecutive epoch(s).


## 3. Training curves

These plots use the saved per-epoch history. They do not retrain the model or evaluate the official test set.

In [3]:
fig, axis = plt.subplots(figsize=(8, 4.5))
axis.plot(history_frame["epoch"], history_frame["training_loss"], marker="o", label="Training loss")
axis.plot(history_frame["epoch"], history_frame["validation_loss"], marker="o", label="Validation loss")
axis.set(title="TextCNN loss by epoch", xlabel="Epoch", ylabel="Cross-entropy loss")
axis.legend()
axis.grid(alpha=0.25)
fig.tight_layout()
display(fig)
plt.close(fig)

<Figure size 800x450 with 1 Axes>

In [4]:
fig, axis = plt.subplots(figsize=(8, 4.5))
axis.plot(history_frame["epoch"], history_frame["validation_f1"], marker="o", color="tab:green")
axis.axvline(metadata["best_epoch"], color="tab:red", linestyle="--", label=f"Best checkpoint: epoch {metadata['best_epoch']}")
axis.set(title="Validation F1 by epoch", xlabel="Epoch", ylabel="F1 score", ylim=(0, 1))
axis.legend()
axis.grid(alpha=0.25)
fig.tight_layout()
display(fig)
plt.close(fig)

<Figure size 800x450 with 1 Axes>

## 4. CPU checkpoint loading and validation examples

The predictor uses the saved vocabulary and the same Phase 4 text encoder. The example reviews below come from the deterministic validation split of the labeled training data. Actual labels are shown for inspection only; no required prediction is assumed.

In [5]:
from deeptext.data.dataset import load_imdb_split, split_training_data
from deeptext.inference import load_textcnn_predictor

predictor = load_textcnn_predictor(RUN_DIRECTORY, device="cpu")
assert next(predictor.model.parameters()).device.type == "cpu"
assert predictor.vocabulary.size == metadata["preprocessing"]["actual_vocabulary_size"]

training_records = load_imdb_split(DATASET_DIRECTORY, "train")
_, validation_records = split_training_data(
    training_records, validation_size=0.2, seed=42
)
assert len(validation_records) == metadata["split"]["validation_rows"]

validation_examples = validation_records.sample(n=8, random_state=42)
rows = []
for _, review in validation_examples.iterrows():
    prediction = predictor.predict(review["text"])
    rows.append({
        "actual_label": "positive" if review["label"] == 1 else "negative",
        "predicted_label": prediction["sentiment"],
        "negative_probability": prediction["negative_probability"],
        "positive_probability": prediction["positive_probability"],
    })
    assert np.isfinite(prediction["negative_probability"])
    assert np.isfinite(prediction["positive_probability"])
    assert np.isclose(
        prediction["negative_probability"] + prediction["positive_probability"],
        1.0, atol=1e-6
    )

display(pd.DataFrame(rows))
print(f"Loaded checkpoint on {next(predictor.model.parameters()).device}; official test split was not loaded.")

,actual_label,predicted_label,negative_probability,positive_probability
0,negative,negative,0.996673,0.003327
1,positive,negative,0.713946,0.286054
2,positive,positive,0.009329,0.990671
3,positive,positive,0.050919,0.949081
4,positive,positive,0.136154,0.863846
5,negative,negative,0.935004,0.064996
6,positive,positive,0.045226,0.954774
7,positive,positive,0.003081,0.996919


Loaded checkpoint on cpu; official test split was not loaded.


## 5. Result and limitations

The best saved checkpoint is epoch 2 with validation accuracy 84.92%, precision 81.93%, recall 89.60%, and F1 85.59%. Early stopping ended the run after epoch 5. These are results from one validation run; the notebook does not claim official test performance or guaranteed predictions.

The checkpoint, vocabulary, history, and full configuration are stored in the experiment directory printed above. Training is deliberately not triggered when this notebook opens or reruns.